# Phase 2 · Resolve pathogen–host pairs and prepare PubMed queries

This phase expands the configured selectors into a saved, explicit pair list before acquisition. PubMed query construction is versioned and pair-aware. A preflight reports pair counts and query previews without making network calls.

Live PubMed search/fetch must be run through a configured acquisition adapter. The current V4 package provides the query builder and normalized run contract; the adapter boundary below is deliberately separate.

In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'pathogen_atlas_v4').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'pathogen_atlas_v4').is_dir():
    raise RuntimeError('Open this notebook from the V4 repository')
sys.path.insert(0, str(PROJECT_ROOT / 'notebooks'))
from _shared import configure_notebook, configured_run_id, log_step
RUN_ID = configured_run_id(PROJECT_ROOT)
ROOT, LOG, LOG_PATH = configure_notebook('02_runs_and_pubmed', RUN_ID)
print('Project root:', ROOT, '| log:', LOG_PATH)

2026-11-02 11:29:45,937 INFO Notebook session started at 2026-11-02T11:29:45.937113+00:00


2026-11-02 11:29:45,937 INFO Project root: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4


2026-11-02 11:29:45,937 INFO Run log: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4/data/v4/logs/pilot-v1/02_runs_and_pubmed.log


Project root: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4 | log: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4/data/v4/logs/pilot-v1/02_runs_and_pubmed.log


## Step 1 · Load configuration and taxonomy references

In [2]:
import yaml, pandas as pd
from IPython.display import display
from pathogen_atlas_v4.runs import resolve_run, save_resolved_run
from pathogen_atlas_v4.pubmed import build_pair_query
CONFIG = yaml.safe_load((ROOT / 'configs/pilot_v1.yaml').read_text())
TAXONOMY_ROOT = sorted((ROOT / 'data/v4/staging').glob('*/taxonomy'))[-1]
hosts = pd.read_parquet(TAXONOMY_ROOT / 'hosts.parquet')
pathogens = pd.read_parquet(TAXONOMY_ROOT / 'pathogens.parquet')
LOG.info('Inputs loaded: hosts=%d pathogens=%d config=%s', len(hosts), len(pathogens), CONFIG['run']['run_id'])
display(CONFIG['run'])

2026-11-02 11:29:46,424 INFO Inputs loaded: hosts=5 pathogens=142201 config=pilot-v1


{'run_id': 'pilot-v1',
 'config_version': 'pilot-config-v1',
 'selection_mode': 'smoke_test',
 'selection_seed': 'v4-pilot-v1',
 'host_ids': ['ncbi:9913'],
 'pathogen_ids': ['bacteria:c714cb9064025c2e9256'],
 'pair_generation': 'explicit_pairs',
 'pairs': [{'pathogen_id': 'bacteria:c714cb9064025c2e9256',
   'host_id': 'ncbi:9913',
   'relationship_context': 'disease'}]}

## Step 2 · Resolve the exact pair set

In [3]:
host_groups = {key: group.host_id.astype(str).tolist() for key, group in hosts.groupby('host_group')}
pathogen_groups = {key: group.pathogen_id.astype(str).tolist() for key, group in pathogens.groupby('pathogen_group')}
resolved = resolve_run(CONFIG, known_host_ids=set(hosts.host_id.astype(str)),
                       known_pathogen_ids=set(pathogens.pathogen_id.astype(str)),
                       host_group_ids=host_groups, pathogen_group_ids=pathogen_groups)
RESOLVED_PATH = ROOT / 'data/v4/runs' / resolved.run_id / 'resolved_config.json'
save_resolved_run(CONFIG, resolved, RESOLVED_PATH)
print('Run:', resolved.run_id, '| generation rule:', resolved.pair_generation)
print('Hosts:', len(resolved.host_ids), '| pathogens:', len(resolved.pathogen_ids), '| pairs:', len(resolved.pairs))
print('Resolved configuration:', RESOLVED_PATH)
LOG.info('Resolved and saved run: hosts=%d pathogens=%d pairs=%d hash=%s', len(resolved.host_ids), len(resolved.pathogen_ids), len(resolved.pairs), resolved.resolved_config_hash)

2026-11-02 11:29:46,453 INFO Resolved and saved run: hosts=1 pathogens=1 pairs=1 hash=fcf335d52396fff80384c7e69f48bb55bde9531f328252f6adb643ef1ab1b25a


Run: pilot-v1 | generation rule: explicit_pairs
Hosts: 1 | pathogens: 1 | pairs: 1
Resolved configuration: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4/data/v4/runs/pilot-v1/resolved_config.json


## Step 3 · Build and inspect pair-aware query text

In [4]:
host_rows = hosts.set_index('host_id')
pathogen_rows = pathogens.set_index('pathogen_id')
PREVIEW_ROWS = []
for pathogen_id, host_id in resolved.pairs[:10]:
    pathogen = pathogen_rows.loc[pathogen_id]
    host = host_rows.loc[host_id]
    host_aliases = host.get('aliases', [])
    if hasattr(host_aliases, 'tolist'):
        host_aliases = host_aliases.tolist()
    host_aliases = list(host_aliases) if host_aliases is not None else []
    query = build_pair_query([pathogen.accepted_name], [host.accepted_name, *host_aliases],
                             strategy_version=CONFIG['acquisition']['query_strategy_version'])
    PREVIEW_ROWS.append({'pathogen_id': pathogen_id, 'pathogen': pathogen.accepted_name,
                         'host_id': host_id, 'host': host.accepted_name, **query})
display(pd.DataFrame(PREVIEW_ROWS))
LOG.info('Generated %d pair query previews using strategy %s', len(PREVIEW_ROWS), CONFIG['acquisition']['query_strategy_version'])

,pathogen_id,pathogen,host_id,host,query,query_hash,query_version
0,bacteria:c714cb9064025c2e9256,Sulfurospirillum multivorans,ncbi:9913,Bos taurus,"(""Sulfurospirillum multivorans""[Title/Abstract...",e32907254bb1852a0129dbbada4cb254e09c8e3a67eef4...,pathogen-host-title-abstract-v1


2026-11-02 11:29:46,466 INFO Generated 1 pair query previews using strategy pathogen-host-title-abstract-v1


## Step 4 · Preflight counts and acquisition readiness

In [5]:
preflight = {'pairs_to_search': len(resolved.pairs), 'distinct_pathogens': len(resolved.pathogen_ids),
             'distinct_hosts': len(resolved.host_ids), 'query_strategy_version': CONFIG['acquisition']['query_strategy_version'],
             'network_calls_made': 0, 'adapter_available': False}
display(preflight)
LOG.info('PubMed preflight: %s', preflight)

{'pairs_to_search': 1,
 'distinct_pathogens': 1,
 'distinct_hosts': 1,
 'query_strategy_version': 'pathogen-host-title-abstract-v1',
 'network_calls_made': 0,
 'adapter_available': False}

2026-11-02 11:29:46,469 INFO PubMed preflight: {'pairs_to_search': 1, 'distinct_pathogens': 1, 'distinct_hosts': 1, 'query_strategy_version': 'pathogen-host-title-abstract-v1', 'network_calls_made': 0, 'adapter_available': False}


## Step 5 · Search and fetch selected literature (adapter step)

In [6]:
RUN_ACQUISITION = False
PUBMED_ADAPTER = None  # supply an adapter that checkpoints pair-scoped PMID results and retrieval state
if RUN_ACQUISITION:
    if PUBMED_ADAPTER is None:
        raise RuntimeError('Configure a PubMed acquisition adapter before enabling network calls.')
    raise NotImplementedError('Adapter integration owns request pacing, retries, checkpoints, and resume logging.')
else:
    print('Preflight complete; no PubMed requests were made.')
    LOG.info('Acquisition skipped: RUN_ACQUISITION=False')

2026-11-02 11:29:46,472 INFO Acquisition skipped: RUN_ACQUISITION=False


Preflight complete; no PubMed requests were made.


The acquisition adapter should log pair progress, retries, ETA, and checkpoint counts to the shared run log. It must save query text/hash, result PMIDs, fetch status, and sampling choices separately from disease-gate decisions.